#### Setup: catalog, schemas, and tables
Run once before any other notebook to create the Unity Catalog for the Blue Ridge Parkway Closures project pipeline.

**Creates**
- Catalog: `brp`
- Schemas (medallion layers):
  - `brp.bronze`: raw data exactly as ingested
  - `brp.silver`: cleaned and structured parkway road section status
  - `brp.gold`: outputs ready to map and daily summary
- Tables:
  - `brp.bronze.road_status_raw`: saved download and metadata of the parkway road status and closures page (nps.gov)
  - `brp.bronze.mileposts`: saved download of the parkway milepost GeoJSON points (arc.gis)
  - `brp.silver.road_sections_status`: the parkway road status and closures page parsed into road sections, one row per section
  - `brp.silver.mileposts`: geo loacation of each parkway milepost point
  - `brp.gold.section_status`: every road section from silver, plus its line shape so it can be drawn
  - `brp.gold.daily_summary`: miles and percentages by status for the whole parkway and each state.

**Steps**
1. Create the catalog
2. Create the three medallion schemas
3. Create the bronze road status table
4. Create the silver road sections 
5. Create the gold summary tables
6. Check: list catalog, schemas, and tables

**Safe to re-run:** every statement uses `IF NOT EXISTS`, so running it again never changes or deletes project tables. This also means any changes to an existing table's columns or comments will not be applied on a rerun; use `ALTER TABLE` / `COMMENT ON`, or drop and recreate the table if changes are needed.

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- PROJECT SETUP: CATALOG
-- Creates `brp`, the top-level container for everything in the Blue Ridge
-- Parkway closures project.
--
-- Unity Catalog organizes data in three levels:
--   catalog  ->  schema  ->  table / volume
--   brp      ->  bronze  ->  road_status_raw
-- For example: brp.bronze.road_status_raw.
--
-- The bronze, silver and gold schemas are created
-- inside the brp catalog in the next cells.
--
-- IF NOT EXISTS makes this safe to rerun: if the catalog is already there,
-- nothing happens, and nothing inside it is changed or deleted.
--
-- Note: creating a catalog needs the CREATE CATALOG permission on the
-- metastore. If this fails with a permissions error, a workspace admin
-- needs to grant it.
-- ---------------------------------------------------------------------------
CREATE CATALOG IF NOT EXISTS brp
COMMENT 'Blue Ridge Parkway closures project: bronze, silver and gold layers';

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- PROJECT SETUP: SCHEMAS (one per medallion layer)
-- Creates the three schemas inside the brp catalog. Each schema holds one
-- layer of the medallion architecture, and data flows bronze -> silver -> gold:
--
--   bronze  Raw data, exactly as downloaded. Never edited, so later layers
--           can always be rebuilt from it.
--
--   silver  Cleaned and structured: parsed into proper columns and types,
--           with all values standardized.
--
--   gold    Summaries built from silver, ready for mapping and dashboard
-- ---------------------------------------------------------------------------

CREATE SCHEMA IF NOT EXISTS brp.bronze
COMMENT 'Raw data, exactly as received from the source';

CREATE SCHEMA IF NOT EXISTS brp.silver
COMMENT 'Cleaned, standardized data parsed from bronze';

CREATE SCHEMA IF NOT EXISTS brp.gold
COMMENT 'Aggregated datasets for mapping and dashboard';

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- BRONZE TABLE: brp.bronze.road_status_raw
-- Download of the NPS road status page.
-- Keeps the raw HTML plus add download metadata, so silver can always be
-- rebuilt from here.
-- ---------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS brp.bronze.road_status_raw (
  source      STRING    COMMENT 'Which source this came from, e.g. road_status_page',
  url         STRING    COMMENT 'Address that was downloaded',
  run_date    DATE      COMMENT 'Pipeline day in America/New_York EST',
  fetched_at  TIMESTAMP COMMENT 'Exact download time (UTC)',
  http_status INT       COMMENT 'Server reply code, 200 = OK',
  sha256      STRING    COMMENT 'Fingerprint of the raw bytes; used to identify a status change',
  raw_html    STRING    COMMENT 'Full page HTML as received'
)
COMMENT 'Bronze: a raw snapshot of the downloaded parkway status page';

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- BRONZE: MILEPOSTS RAW (reference geometry)
-- The NPS milepost layer exactly as downloaded from ArcGIS
-- Stored as GeoJSON text, plus metatdata about the download.
-- The latest download only (overwritten when refreshed).
-- Source: NPS SEAC_BLRI_Mileposts ArcGIS feature service
-- Used by: brp.silver.mileposts
-- ---------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS brp.bronze.mileposts_raw (
  -- When and where the data came from
  fetched_at     TIMESTAMP COMMENT 'Exact download time (UTC)',
  url            STRING    COMMENT 'ArcGIS query address the points were downloaded from',
  http_status    INT       COMMENT 'Server reply code of the last page request; 200 = OK',

  -- What was received
  feature_count  INT       COMMENT 'Number of milepost features downloaded',
  sha256         STRING    COMMENT 'Fingerprint of raw_geojson; changes if the source data changes',
  raw_geojson    STRING    COMMENT 'All downloaded features as one GeoJSON FeatureCollection, unchanged'
)
COMMENT 'Bronze: NPS milepost points as downloaded (GeoJSON), latest download only';

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- PROJECT SETUP: SILVER TABLE
-- The road status page parsed into clean columns: one row per parkway section 
-- Built from brp.bronze.road_status_raw.
-- Columns must match SILVER_SCHEMA in the silver notebook.
-- ---------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS brp.silver.road_sections_status (
  -- When the data was captured
  run_date      DATE      COMMENT 'Pipeline day in America/New_York EST',
  fetched_at    TIMESTAMP COMMENT 'Exact download time (UTC); links to the bronze row',
  nps_as_of     STRING    COMMENT 'NPS "Road status as of" text from the HTML page',

  -- Where the section is
  state         STRING    COMMENT 'VA or NC',
  section_order INT       COMMENT 'Row order on the page (mileposts increase north to south)',
  mp_start      DOUBLE    COMMENT 'Lower milepost of the section',
  mp_end        DOUBLE    COMMENT 'Higher milepost of the section',
  length_mi     DOUBLE    COMMENT 'Section length in miles (mp_end - mp_start)',
  crossroads    STRING    COMMENT 'Roads or landmarks at each end of the section',

  -- What the status is
  status_raw    STRING    COMMENT 'Status exactly as written by NPS',
  notes         STRING    COMMENT 'Notes / maintenance information column',
  status        STRING    COMMENT 'Classified: open, reduced, closed, unknown',
  gated         BOOLEAN   COMMENT 'False when NPS lists the section as Ungated'
)
COMMENT 'Silver: Blue Ridge Parkway road status by mileposts section, one row per section';

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- PROJECT SETUP: SILVER MILEPOSTS (reference table)
-- One row per milepost along the parkway (0 to 469), each with its
-- map coordinates. Loaded from ArcGIS milepost data.
--
-- Used to place road sections on a map: a section like 291.8 - 305.1 is
-- drawn by looking up the points for its start and end mileposts, and
-- estimating positions between mileposts where needed.
--
-- Reference data: it is unlikely to change, unlike the road status table.
-- ---------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS brp.silver.mileposts (
  milepost    DOUBLE    COMMENT 'Parkway milepost (whole miles, 0 to 469); DOUBLE to match mp_start / mp_end in road_sections_status',
  longitude   DOUBLE    COMMENT 'Longitude (east-west position), WGS84 / EPSG:4326, the standard GPS coordinate system',
  latitude    DOUBLE    COMMENT 'Latitude (north-south position), WGS84 / EPSG:4326',
  fetched_at  TIMESTAMP COMMENT 'When the points were downloaded (UTC)'
)
COMMENT 'Silver reference: NPS Blue Ridge Parkway milepost points (one per milepost)';

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- GOLD: SECTION STATUS (map-ready)
-- Every road section from silver, plus its line shape so it can be drawn
-- on a map and colored by status.
-- One row per section.
-- Built from: brp.silver.road_sections_status + brp.silver.mileposts
-- ---------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS brp.gold.section_status (
  -- When the data was captured
  run_date          DATE      COMMENT 'Pipeline day in America/New_York EST',
  fetched_at        TIMESTAMP COMMENT 'Exact download time (UTC); links to the silver and bronze rows',
  nps_as_of         STRING    COMMENT 'When NPS last updated the page ("Road status as of" text)',

  -- Where the section is
  state             STRING    COMMENT 'VA or NC',
  section_order     INT       COMMENT 'Row order on the page (mileposts increase north to south)',
  mp_start          DOUBLE    COMMENT 'Lower milepost of the section',
  mp_end            DOUBLE    COMMENT 'Higher milepost of the section',
  length_mi         DOUBLE    COMMENT 'Section length in miles',
  crossroads        STRING    COMMENT 'Roads or landmarks at each end of the section',

  -- What the status is
  status            STRING    COMMENT 'Classified: open, reduced, closed, unknown',
  status_raw        STRING    COMMENT 'Status exactly as written by NPS',
  notes             STRING    COMMENT 'NPS notes / maintenance information',
  gated             BOOLEAN   COMMENT 'False when NPS lists the section as Ungated',

  -- Map shape
  point_count       INT       COMMENT 'How many points make up the line',
  geometry_geojson  STRING    COMMENT 'The parkway section line as GeoJSON text: [longitude, latitude] points in GPS coordinates (WGS84 / EPSG:4326)'
)
COMMENT 'Gold: map-ready road sections with status and line shape';

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- GOLD: DAILY SUMMARY (KPI cards)
-- Miles and percentages by status for the whole parkway and each state.
-- Three rows per run: scope = ALL, VA, NC.
-- Built from: brp.gold.section_status
-- ---------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS brp.gold.daily_summary (
  -- When the data was captured
  run_date       DATE      COMMENT 'Pipeline day in America/New_York EST',
  fetched_at     TIMESTAMP COMMENT 'Exact download time (UTC)',
  nps_as_of      STRING    COMMENT 'When NPS last updated the page ("Road status as of" text)',

  -- What the row covers
  scope          STRING    COMMENT 'ALL (whole parkway), VA, or NC',
  sections       INT       COMMENT 'Number of sections in this scope',

  -- Miles by status
  miles_total    DOUBLE    COMMENT 'Total miles in this scope',
  miles_open     DOUBLE    COMMENT 'Miles fully open',
  miles_reduced  DOUBLE    COMMENT 'Miles open with lane closures or delays',
  miles_closed   DOUBLE    COMMENT 'Miles closed',
  miles_unknown  DOUBLE    COMMENT 'Miles with a status the rules do not recognize',

  -- Percentages of total miles (0-100)
  pct_open       DOUBLE    COMMENT 'Percent of miles fully open',
  pct_passable   DOUBLE    COMMENT 'Percent of miles open or reduced (drivable)'
)
COMMENT 'Gold: parkway status totals for the whole parkway and each state';

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- SETUP CHECK: WHAT'S IN THE brp CATALOG
-- Lists every schema and the tables inside it, to confirm the medallion
-- setup (bronze, silver, gold) was created correctly.
--
-- Expected: bronze, silver and gold all appear. 
-- ---------------------------------------------------------------------------
SELECT
    schemata.catalog_name AS table_catalog,   -- always brp here
    schemata.schema_name  AS table_schema,    -- bronze, silver, gold
    tables.table_name,                        -- a table in that schema (NULL if it has none)
    tables.table_type                         -- MANAGED (Databricks stores the data)

-- information_schema is a built-in set of read-only views describing what's
-- in the catalog: `schemata` lists schemas, `tables` lists tables.
FROM brp.information_schema.schemata

-- LEFT JOIN keeps every schema even if it has no tables.
LEFT JOIN brp.information_schema.tables
    ON  schemata.catalog_name = tables.table_catalog
    AND schemata.schema_name  = tables.table_schema

-- Hide the two schemas Databricks creates automatically in every catalog:
--   information_schema = the built-in metadata views used above
--   default            = an empty starter schema this project doesn't use
WHERE schemata.schema_name NOT IN ('information_schema', 'default')

-- Group the list by schema, then alphabetically by table.
ORDER BY schemata.schema_name, tables.table_name